# Model: LightGBM

Owner: **Arman**

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score

from lightgbm import LGBMRegressor

MODEL_NAME = "lightgbm"

In [ ]:
# using the shared train/validation/test sets

DATA_DIR = Path("../../data/NSW")
RESULTS_DIR = Path("../../results")
RESULTS_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "nsw_train.csv", parse_dates=["DATETIME"])
validation = pd.read_csv(DATA_DIR / "nsw_validation.csv", parse_dates=["DATETIME"])
test = pd.read_csv(DATA_DIR / "nsw_test.csv", parse_dates=["DATETIME"])

print(train.shape, validation.shape, test.shape)

In [ ]:
# TEMPERATURE/radiation are same-day actuals so not usable, only the day_before lags are
# (see model_random_forest.ipynb - team agreed to drop the forecast_* columns too)

TARGET = "TOTALDEMAND"
DROP_COLS = ["DATETIME", TARGET, "TEMPERATURE", "radiation", "forecast_closest", "forecast_12hr_prior", "forecast_dayprior"]
FEATURES = [c for c in train.columns if c not in DROP_COLS]

len(FEATURES)

In [ ]:
# TODO: train LGBMRegressor on train[FEATURES]/train[TARGET], use validation to pick hyperparameters
# see model_random_forest.ipynb for an example: loop over a param grid, fit on train, score against
# validation with mean_squared_error, keep whichever setting scores best

model = None

In [ ]:
# TODO: predict on test now that the model + params are settled using validation above

predictions = None

In [ ]:
def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2 = r2_score(y_true, y_pred)
    metrics = {"model": model_name, "rmse": rmse, "mae": mae, "mape_pct": mape, "r2": r2}
    print(metrics)
    return metrics


metrics = evaluate(test[TARGET], predictions, MODEL_NAME)

In [ ]:
# saving prediction performance

pd.Series(predictions, index=test["DATETIME"], name=MODEL_NAME).to_csv(RESULTS_DIR / f"{MODEL_NAME}_predictions.csv")

comparison_path = RESULTS_DIR / "model_comparison.csv"
this_run = pd.DataFrame([metrics])

if comparison_path.exists():
    existing = pd.read_csv(comparison_path)
    existing = existing[existing["model"] != MODEL_NAME]
    this_run = pd.concat([existing, this_run], ignore_index=True)

this_run.to_csv(comparison_path, index=False)
this_run